In [2]:
import polars as pl 
from dbconfig import engine
print('Environment ready!')

Environment ready!


In [24]:
from sqlalchemy import text

In [15]:
pl.Config.set_tbl_rows(100)          # rows shown
pl.Config.set_tbl_cols(50)           # columns shown
pl.Config.set_fmt_str_lengths(200)   # string truncation width
pl.Config.set_tbl_width_chars(200)   # total table width

polars.config.Config

In [8]:
pl.read_database(
        query = """
        select count(*),
        count(distinct (order_id, payment_sequential)) as distinct_rows
        from raw.payments;
        """, connection = engine
        )

count,distinct_rows
i64,i64
103886,103886


In [10]:
pl.read_database(
        query = """
        select count(*),
        count(distinct(review_id, order_id)) as distinct_rows
        from raw.reviews;
        """, connection = engine
        )

count,distinct_rows
i64,i64
99224,99224


In [11]:
pl.read_database(
        query = """
        select schema_name
        from information_schema.schemata
        order by schema_name;
        """, connection = engine
        )

schema_name
str
"""analytics"""
"""clean"""
"""information_schema"""
"""pg_catalog"""
"""pg_toast"""
"""public"""
"""raw"""


In [12]:
pl.read_database(
        query = """
        select table_schema,
        table_name
        from information_schema.tables 
        where table_schema in ('raw', 'clean', 'analytics')
        order by table_schema, table_name;
        """, connection = engine
        )

table_schema,table_name
str,str
"""raw""","""category_translation"""
"""raw""","""customers"""
"""raw""","""geolocation"""
"""raw""","""order_items"""
"""raw""","""orders"""
"""raw""","""payments"""
"""raw""","""products"""
"""raw""","""reviews"""
"""raw""","""sellers"""


In [16]:
pl.read_database(
        query = """
        select table_name,
        column_name, data_type
        from information_schema.columns
        where table_schema = 'raw'   
        order by table_name, ordinal_position;
        """, connection = engine
        )

table_name,column_name,data_type
str,str,str
"""category_translation""","""product_category_name""","""text"""
"""category_translation""","""product_category_name_english""","""text"""
"""customers""","""customer_id""","""text"""
"""customers""","""customer_unique_id""","""text"""
"""customers""","""customer_zip_code_prefix""","""bigint"""
"""customers""","""customer_city""","""text"""
"""customers""","""customer_state""","""text"""
"""geolocation""","""geolocation_zip_code_prefix""","""bigint"""
"""geolocation""","""geolocation_lat""","""double precision"""


In [33]:
with engine.connect() as conn:
    conn.execute(
        text("""
        create table clean.categories(
            product_category_name text primary key,
            product_category_name_english text
            );"""
        ))
    conn.commit()

In [34]:
with engine.connect() as conn:
    conn.execute(
        text("""
             insert into clean.categories
             select *
             from raw.category_translation;
             """)
        )
    conn.commit()

In [35]:
pl.read_database(
        query = """
             select table_schema, 
             table_name
             from information_schema.tables 
             where table_schema = 'clean';
             """, connection = engine
        )

table_schema,table_name
str,str
"""clean""","""categories"""


In [37]:
pl.read_database(
        query = """
        select count(*)
        from clean.categories;
        """, connection = engine
        )

count
i64
71


In [38]:
with engine.connect() as conn:
    conn.execute(
        text("""
        create table clean.customers(
            customer_id text primary key,
            customer_unique_id text not null,
            customer_zip_code_prefix bigint not null,
            customer_city text not null,
            customer_state text not null
        );
        """)
    )
    conn.commit()

In [39]:
with engine.connect() as conn:
    conn.execute(
        text("""
        insert into clean.customers
        select *
        from raw.customers;
        """)
    )
    conn.commit()

In [40]:
pl.read_database("""
select count(*)
from clean.customers
""", engine)

count
i64
99441


In [41]:
with engine.connect() as conn:
    conn.execute(
        text("""
        create table clean.sellers(
            seller_id text primary key,
            seller_zip_code_prefix bigint not null,
            seller_city text not null,
            seller_state text not null
        );
        """)
    )
    conn.commit()

In [42]:
with engine.connect() as conn:
    conn.execute(
        text("""
        insert into clean.sellers
        select *
        from raw.sellers;
        """)
    )
    conn.commit()

In [43]:
pl.read_database("""
select count(*)
from clean.sellers
""", engine)

count
i64
3095


In [44]:
with engine.connect() as conn:
    conn.execute(
        text("""
        create table clean.products(
            product_id text primary key,
            product_category_name text,
            product_name_lenght bigint,
            product_description_lenght bigint,
            product_photos_qty bigint,
            product_weight_g bigint,
            product_length_cm bigint,
            product_height_cm bigint,
            product_width_cm bigint
        );
        """)
    )
    conn.commit()

In [45]:
with engine.connect() as conn:
    conn.execute(
        text("""
        insert into clean.products
        select *
        from raw.products;
        """)
    )
    conn.commit()

In [46]:
pl.read_database("""
select count(*)
from clean.products
""", engine)

count
i64
32951


In [47]:
pl.read_database(
        query = """
        select table_name
        from information_schema.tables
        where table_schema = 'clean';
        """, connection = engine
        )

table_name
str
"""categories"""
"""customers"""
"""sellers"""
"""products"""


In [48]:
with engine.connect() as conn:
    conn.execute(
        text("""
        create table clean.payments(
            order_id text,
            payment_sequential bigint,
            payment_type text not null,
            payment_installments bigint not null,
            payment_value double precision not null,
            primary key(order_id, payment_sequential)
        );
        """)
    )
    conn.commit()

In [49]:
with engine.connect() as conn:
    conn.execute(
        text("""
        insert into clean.payments
        select *
        from raw.payments;
        """)
    )
    conn.commit()

In [50]:
pl.read_database("""
select count(*)
from clean.payments
""", engine)

count
i64
103886


In [51]:
with engine.connect() as conn:
    conn.execute(
        text("""
        create table clean.orders(
            order_id text primary key,
            customer_id text not null,
            order_status text not null,
            order_purchase_timestamp timestamp not null,
            order_approved_at timestamp,
            order_delivered_carrier_date timestamp,
            order_delivered_customer_date timestamp,
            order_estimated_delivery_date timestamp not null
        );
        """)
    )
    conn.commit()

In [52]:
with engine.connect() as conn:
    conn.execute(
        text("""
        insert into clean.orders
        select
            order_id,
            customer_id,
            order_status,
            order_purchase_timestamp::timestamp,
            order_approved_at::timestamp,
            order_delivered_carrier_date::timestamp,
            order_delivered_customer_date::timestamp,
            order_estimated_delivery_date::timestamp
        from raw.orders;
        """)
    )
    conn.commit()

In [53]:
pl.read_database(
        query = """
        select count(*)
        from clean.orders;
        """, connection = engine
        )

count
i64
99441


In [54]:
with engine.connect() as conn:
    conn.execute(
        text("""
        create table clean.reviews(
            review_id text not null,
            order_id text not null,
            review_score bigint not null,
            review_comment_title text,
            review_comment_message text,
            review_creation_date timestamp not null,
            review_answer_timestamp timestamp not null,
            primary key(review_id, order_id)
        );
        """)
    )
    conn.commit()

In [55]:
with engine.connect() as conn:
    conn.execute(
        text("""
        insert into clean.reviews
        select
            review_id,
            order_id,
            review_score,
            review_comment_title,
            review_comment_message,
            review_creation_date::timestamp,
            review_answer_timestamp::timestamp
        from raw.reviews;
        """)
    )
    conn.commit()

In [56]:
pl.read_database(
        query = """
        select count(*)
        from clean.reviews;
        """, connection = engine
        )

count
i64
99224


In [57]:
with engine.connect() as conn:
    conn.execute(
        text("""
        create table clean.order_items(
            order_id text not null,
            order_item_id bigint not null,
            product_id text not null,
            seller_id text not null,
            shipping_limit_date timestamp not null,
            price double precision not null,
            freight_value double precision not null,
            primary key(order_id, order_item_id)
        );
        """)
    )
    conn.commit()

In [58]:
with engine.connect() as conn:
    conn.execute(
        text("""
        insert into clean.order_items
        select
            order_id,
            order_item_id,
            product_id,
            seller_id,
            shipping_limit_date::timestamp,
            price,
            freight_value
        from raw.order_items;
        """)
    )
    conn.commit()

In [59]:
pl.read_database(
        query = """
        select count(*)
        from clean.order_items;
        """, connection = engine
        )

count
i64
112650


In [61]:
pl.read_database(
        query = """
        select table_name,
        column_name,
        data_type
        from information_schema.columns
        where table_schema = 'clean'
        order by table_name, ordinal_position;
        """, connection = engine
        )

table_name,column_name,data_type
str,str,str
"""categories""","""product_category_name""","""text"""
"""categories""","""product_category_name_english""","""text"""
"""customers""","""customer_id""","""text"""
"""customers""","""customer_unique_id""","""text"""
"""customers""","""customer_zip_code_prefix""","""bigint"""
"""customers""","""customer_city""","""text"""
"""customers""","""customer_state""","""text"""
"""order_items""","""order_id""","""text"""
"""order_items""","""order_item_id""","""bigint"""


In [62]:
pl.read_database(
        query = """
        select table_name,
        constraint_type
        from information_schema.table_constraints
        where table_schema = 'clean'
        order by table_name;
        """, connection = engine
        )

table_name,constraint_type
str,str
"""categories""","""PRIMARY KEY"""
"""categories""","""CHECK"""
"""customers""","""CHECK"""
"""customers""","""CHECK"""
"""customers""","""CHECK"""
"""customers""","""CHECK"""
"""customers""","""PRIMARY KEY"""
"""customers""","""CHECK"""
"""order_items""","""CHECK"""


In [63]:
with engine.connect() as conn:
    conn.execute(
        text("""
        alter table clean.orders
        add constraint fk_orders_customer
        foreign key (customer_id)
        references clean.customers(customer_id);
        """)
    )
    conn.commit()

In [69]:
with engine.connect() as conn:
    conn.execute(
        text("""
        alter table clean.products
        add constraint fk_products_category
        foreign key (product_category_name)
        references clean.categories(product_category_name);
        """)
    )
    conn.commit()

In [65]:
pl.read_database("""select distinct p.product_category_name
from clean.products p
left join clean.categories c
    on p.product_category_name = c.product_category_name
where p.product_category_name is not null
  and c.product_category_name is null
order by 1;""", connection = engine)

product_category_name
str
"""pc_gamer"""
"""portateis_cozinha_e_preparadores_de_alimentos"""


In [66]:
pl.read_database("""select *
from raw.category_translation
where product_category_name in (
    'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos'
);""", connection = engine)

product_category_name,product_category_name_english
null,null


In [67]:
pl.read_database("""select
    product_category_name,
    count(*) as products
from clean.products
where product_category_name in (
    'pc_gamer',
    'portateis_cozinha_e_preparadores_de_alimentos'
)
group by 1;""", connection = engine)

product_category_name,products
str,i64
"""pc_gamer""",3
"""portateis_cozinha_e_preparadores_de_alimentos""",10


In [68]:
with engine.connect() as conn:
    conn.execute(
        text("""
        insert into clean.categories
        (product_category_name, product_category_name_english)
        values
        ('pc_gamer', null),
        ('portateis_cozinha_e_preparadores_de_alimentos', null);
        """)
    )
    conn.commit()

In [70]:
with engine.connect() as conn:
    conn.execute(
        text("""
        alter table clean.orders
        add constraint fk_orders_customer
        foreign key (customer_id)
        references clean.customers(customer_id);
        """)
    )
    conn.commit()

ProgrammingError: (psycopg.errors.DuplicateObject) constraint "fk_orders_customer" for relation "orders" already exists
[SQL: 
        alter table clean.orders
        add constraint fk_orders_customer
        foreign key (customer_id)
        references clean.customers(customer_id);
        ]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [71]:
pl.read_database("""SELECT
    tc.table_name,
    tc.constraint_name,
    tc.constraint_type
FROM information_schema.table_constraints tc
WHERE tc.table_schema = 'clean'
ORDER BY tc.table_name, tc.constraint_name;""", connection = engine)

table_name,constraint_name,constraint_type
str,str,str
"""categories""","""16390_16485_1_not_null""","""CHECK"""
"""categories""","""categories_pkey""","""PRIMARY KEY"""
"""customers""","""16390_16492_1_not_null""","""CHECK"""
"""customers""","""16390_16492_2_not_null""","""CHECK"""
"""customers""","""16390_16492_3_not_null""","""CHECK"""
"""customers""","""16390_16492_4_not_null""","""CHECK"""
"""customers""","""16390_16492_5_not_null""","""CHECK"""
"""customers""","""customers_pkey""","""PRIMARY KEY"""
"""order_items""","""16390_16549_1_not_null""","""CHECK"""


In [73]:
pl.read_database("""SELECT
    table_name,
    constraint_name
FROM information_schema.table_constraints
WHERE table_schema = 'clean'
  AND constraint_type = 'FOREIGN KEY'
ORDER BY table_name;""", connection = engine)

table_name,constraint_name
str,str
"""orders""","""fk_orders_customer"""
"""products""","""fk_products_category"""


In [74]:
with engine.connect() as conn:
    conn.execute(
        text("""
        alter table clean.order_items
        add constraint fk_order_items_order
        foreign key (order_id)
        references clean.orders(order_id);
        """)
    )
    conn.commit()

In [75]:
 with engine.connect() as conn:
    conn.execute(
        text("""
        alter table clean.order_items
        add constraint fk_order_items_product
        foreign key (product_id)
        references clean.products(product_id);
        """)
    )
    conn.commit()

In [76]:
with engine.connect() as conn:
    conn.execute(
        text("""
        alter table clean.order_items
        add constraint fk_order_items_seller
        foreign key (seller_id)
        references clean.sellers(seller_id);
        """)
    )
    conn.commit()

In [77]:
with engine.connect() as conn:
    conn.execute(
        text("""
        alter table clean.payments
        add constraint fk_payments_order
        foreign key (order_id)
        references clean.orders(order_id);
        """)
    )
    conn.commit()

In [78]:
with engine.connect() as conn:
    conn.execute(
        text("""
        alter table clean.reviews
        add constraint fk_reviews_order
        foreign key (order_id)
        references clean.orders(order_id);
        """)
    )
    conn.commit()

In [79]:
pl.read_database("""SELECT
    table_name,
    constraint_name
FROM information_schema.table_constraints
WHERE table_schema = 'clean'
  AND constraint_type = 'FOREIGN KEY'
ORDER BY table_name;""", connection = engine)

table_name,constraint_name
str,str
"""order_items""","""fk_order_items_product"""
"""order_items""","""fk_order_items_order"""
"""order_items""","""fk_order_items_seller"""
"""orders""","""fk_orders_customer"""
"""payments""","""fk_payments_order"""
"""products""","""fk_products_category"""
"""reviews""","""fk_reviews_order"""
